# CP2 LastFM — Bucket Waktu Eksplisit

Notebook Colab ini menerapkan skema gap yang disepakati untuk persiapan time embedding. Ia mengunggah sequence item dan gap timestamp hasil preprocessing, memvalidasi alignment, memetakan gap ke bucket, dan menyimpan output serta konfigurasi. **Tidak menjalankan training atau mengubah file CP1.**

Catatan interpretasi: waktu HetRec adalah waktu pemberian tag artis, bukan waktu mendengarkan/scrobble.

Pemisahan bucket: nilai pertama sequence adalah kategori khusus "belum ada histori". Gap nol setelah event pertama adalah kategori lain untuk dua event bertimestamp sama. Ini menghindari mencampur start-of-sequence dengan timestamp tie.


## 1. Upload file di Colab


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
ITEM_FILE=Path("/content/LastFM.txt")
GAP_FILE=Path("/content/LastFM_time.txt")
assert ITEM_FILE.exists() and GAP_FILE.exists(), "Upload LastFM.txt dan LastFM_time.txt"


## 2. Baca file dan validasi


In [ ]:
from collections import OrderedDict, Counter
from pathlib import Path
import hashlib, json, statistics

def read_seq(path):
    out=OrderedDict()
    for no,line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(),1):
        vals=list(map(int,line.split()))
        assert len(vals)>=2, f"Baris kosong/pendek {no}"
        assert vals[0] not in out, f"User ID ganda: {vals[0]}"
        out[vals[0]]=vals[1:]
    return out
items=read_seq(ITEM_FILE); gaps=read_seq(GAP_FILE)
assert list(items)==list(gaps), "Urutan user berbeda"
assert all(len(items[u])==len(gaps[u]) for u in items), "Panjang sequence item/gap berbeda"
assert all(len(seq)==len(set(seq)) for seq in items.values()), "Duplikat item ditemukan"
assert all(seq and seq[0]==0 and min(seq)>=0 for seq in gaps.values()), "Gap pertama harus 0 dan gap tidak negatif"
print("Users:",len(items),"artists:",len({x for s in items.values() for x in s}),"events:",sum(map(len,items.values())))



## 3. Tetapkan skema bucket

Bucket ID: 0 = posisi pertama (belum ada gap histori); 1 = gap tepat 0; 2 = 0–1 jam; 3 = 1 jam–1 hari; 4 = 1–7 hari; 5 = 7–30 hari; 6 = 30–365 hari; 7 = lebih dari 365 hari. Batas atas diperlakukan inklusif.


In [ ]:
HOUR=3600
DAY=24*HOUR
WEEK=7*DAY
MONTH=30*DAY
YEAR=365*DAY

def encode_gap(g, first_position=False):
    if first_position: return 0
    if g==0: return 1
    if g<=HOUR: return 2
    if g<=DAY: return 3
    if g<=WEEK: return 4
    if g<=MONTH: return 5
    if g<=YEAR: return 6
    return 7

bucket_seqs=OrderedDict()
for uid, seq in gaps.items():
    bucket_seqs[uid]=[encode_gap(g, first_position=(i==0)) for i,g in enumerate(seq)]
counts=Counter(b for seq in bucket_seqs.values() for b in seq)
print("All-position bucket counts:",dict(sorted(counts.items())))
# Adjacent gaps only, excluding first-position marker
adjacent=Counter(b for seq in bucket_seqs.values() for b in seq[1:])
print("Adjacent-gap bucket counts:",dict(sorted(adjacent.items())))



## 4. Tinjau gap ekstrem dan kategori terluar


In [ ]:
extreme=[(uid,i,g,round(g/DAY,2)) for uid,seq in gaps.items() for i,g in enumerate(seq) if i>0 and g>YEAR]
print("Gaps > 1 year:",len(extreme))
print("Extreme examples (user, position, seconds, days):",extreme[:20])
assert all(bucket_seqs[u][i]==7 for u,i,_,_ in extreme)
print("All >1-year gaps map to bucket 7: PASS")


## 5. Periksa penggunaan histori tanpa target

Satu contoh memakai item dan bucket pada posisi yang sama untuk histori, lalu memisahkan item berikutnya sebagai target. Bucket pada posisi target tidak diberikan sebagai input. Ini hanya demonstrasi alignment; split evaluasi final harus mengikuti loader CP1.


In [ ]:
MAX_LEN=50
uid=next(iter(items))
seq=items[uid]; bseq=bucket_seqs[uid]
hist_items=seq[:-1][-MAX_LEN:]
hist_buckets=bseq[:-1][-MAX_LEN:]
target=seq[-1]
assert len(hist_items)==len(hist_buckets)
assert len(hist_items)==len(seq[:-1][-MAX_LEN:])
print("Example user:",uid,"history length:",len(hist_items),"target:",target)
print("History item IDs:",hist_items[:12])
print("History time bucket IDs:",hist_buckets[:12])
print("Target-position bucket excluded from input:",bseq[-1])


## 6. Simpan bucket dan manifest


In [ ]:
OUT=Path("/content/cp2_explicit_buckets")
OUT.mkdir(exist_ok=True)
BUCKET_FILE=OUT/"LastFM_cp2_time_buckets.txt"
with BUCKET_FILE.open("w",encoding="utf-8",newline="\n") as f:
    for uid,seq in bucket_seqs.items(): f.write(str(uid)+" "+" ".join(map(str,seq))+"\n")
config={
 "bucket_ids":{"0":"first event / no prior history","1":"zero gap / equal adjacent timestamp","2":"0 < gap <= 1 hour","3":"1 hour < gap <= 1 day","4":"1 day < gap <= 1 week","5":"1 week < gap <= 30 days","6":"30 days < gap <= 365 days","7":"gap > 365 days"},
 "units":"seconds","normalization":"none; categorical embedding IDs",
 "semantics":"HetRec artist-tag event time, not listening/scrobble time",
 "target_leakage_rule":"for target at position t, pass only item/time positions before t",
 "counts":{"users":len(items),"artists":len({x for s in items.values() for x in s}),"interactions":sum(map(len,items.values())),"bucket_counts_all_positions":dict(sorted(counts.items())),"bucket_counts_adjacent_gaps":dict(sorted(adjacent.items()))},
 "input_sha256":{"items":hashlib.sha256(ITEM_FILE.read_bytes()).hexdigest(),"gaps":hashlib.sha256(GAP_FILE.read_bytes()).hexdigest()}
}
CONFIG_FILE=OUT/"CP2_time_bucket_config.json"
CONFIG_FILE.write_text(json.dumps(config,indent=2),encoding="utf-8")
print(json.dumps(config,indent=2))
files.download(str(BUCKET_FILE)); files.download(str(CONFIG_FILE))


## Tahap sesudah notebook ini

1. Simpan file bucket dan konfigurasi.
2. Cocokkan sequence item lengkap dengan CP1 atau sumber preprocessing; hitungan agregat yang sama saja belum cukup.
3. Buka source BSARec CP1, lalu tambahkan input bucket ke Dataset/DataLoader dan embedding item pada salinan CP2. Jangan ubah objective BSARec.
4. Pastikan train/validation/test split, mask histori, seed, budget, dan evaluasi identik. Baseline CP1 perlu dijalankan ulang hanya bila perbandingan berpasangan memakai runtime/config yang sama dan memang diperlukan.
5. Baru training baseline dan varian time-aware; laporkan delta per seed dan caveat timestamp tagging.

Notebook ini hanya menyiapkan bucket. Integrasi model belum disertakan karena source loader/model CP1 belum tersedia untuk dicocokkan.
